<a href="https://colab.research.google.com/github/Rasmy-r7/Research/blob/main/mozillacore_Preprocess.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install openpyxl -q

import pandas as pd
import re
from google.colab import files

print("✅ Ready!")

✅ Ready!


In [ ]:
print("📁 Upload Core_bugs.csv")
uploaded = files.upload()

df = pd.read_csv('Core_bugs.csv', low_memory=False)
print(f"✅ Loaded: {df.shape}")
print("\nOriginal Priority distribution:")
print(df['Priority'].value_counts(dropna=False))

📁 Upload Core_bugs.csv


Saving Core_bugs.csv to Core_bugs.csv
✅ Loaded: (85673, 8)

Original Priority distribution:
Priority
--    34722
P5    26554
P3    14321
P2     6317
P1     3470
P4      289
Name: count, dtype: int64


In [ ]:
# Mozilla Core is Bugzilla-based (same as Firefox)
# P1/P2 → High, P3 → Medium, P4/P5 → Low
# NaN, --, and corrupted values → drop

bugzilla_map = {
    'P1': 'High',
    'P2': 'High',
    'P3': 'Medium',
    'P4': 'Low',
    'P5': 'Low',
    '--': None
}

# Strip whitespace first to handle any padded values
df['Priority'] = df['Priority'].astype(str).str.strip()

# Map — anything not in map (NaN, corrupted rows) becomes None
df['priority'] = df['Priority'].map(bugzilla_map)

# Drop unmapped rows (NaN, --, corrupted)
before = len(df)
df = df[df['priority'].notna()].copy()
df = df.reset_index(drop=True)
after = len(df)

print(f"✅ After fixing Priority: {df.shape}")
print(f"   Dropped {before - after} rows (NaN / -- / corrupted)")
print()
print(df['priority'].value_counts())

✅ After fixing Priority: (50951, 9)
   Dropped 34722 rows (NaN / -- / corrupted)

priority
Low       26843
Medium    14321
High       9787
Name: count, dtype: int64


In [ ]:
df['text'] = (
    df['Summary'].fillna('').str.strip() + ' ' +
    df['Description'].fillna('').str.strip()
)

print(f"✅ Text column created!")
print(f"Avg words (before clean): {df['text'].str.split().str.len().mean():.0f}")
print(f"Max words (before clean): {df['text'].str.split().str.len().max()}")

✅ Text column created!
Avg words (before clean): 398
Max words (before clean): 6710


In [ ]:
def clean_text(text):
    text = str(text)

    # Remove log lines [task 2020-01-02T14:00:26Z]
    text = re.sub(
        r'\[task \d{4}-\d{2}-\d{2}T[\d:.]+Z\].*?(?=\[task|\Z)',
        '', text, flags=re.DOTALL
    )
    # Remove Bugzilla boilerplate
    text = re.sub(
        r'(User Agent|Build Identifier|Filed by|Parsed log|Full log)\s*:.*?(?=\n|$)',
        '', text, flags=re.IGNORECASE
    )
    # Remove test log lines
    text = re.sub(
        r'(TEST-PASS|TEST-OK|TEST-FAIL|TEST-UNEXPECTED|GECKO\(\d+\)|INFO\s*-)\s*.*?(?=\n|$)',
        '', text, flags=re.IGNORECASE
    )
    # Remove C++ stack traces (Mozilla Core is C++)
    text = re.sub(
        r'#\d+\s+0x[0-9a-fA-F]+.*?(?=\n|$)',
        '', text, flags=re.IGNORECASE
    )
    # Remove C++ function signatures with :: and *
    text = re.sub(
        r'[\w:]+\*?\s+[\w:]+\([^)]*\)\s*(\[.*?\])?\s*\+\s*0x[0-9a-fA-F]+',
        '', text
    )
    # Remove memory stat lines
    text = re.sub(r'MEMORY STAT.*?(?=\n|$)', '', text, flags=re.IGNORECASE)
    # Remove URLs
    text = re.sub(r'http[s]?://\S+', '', text)
    text = re.sub(r'www\.\S+', '', text)
    # Remove file paths
    text = re.sub(
        r'[\w/\-\.]+\.(js|cpp|h|py|java|html|css|ts|json|xml|md|log|yaml|conf|sh)\b',
        '', text, flags=re.IGNORECASE
    )
    # Remove code blocks
    text = re.sub(r'```.*?```', ' ', text, flags=re.DOTALL)
    text = re.sub(r'`[^`\n]+`', ' ', text)
    # Remove markdown bold/italic
    text = re.sub(r'\*\*([^*]+)\*\*', r'\1', text)
    text = re.sub(r'\*([^*\n]+)\*', r'\1', text)
    # Remove markdown links [text](url)
    text = re.sub(r'\[([^\]]+)\]\([^\)]*\)', r'\1', text)
    # Remove hex values
    text = re.sub(r'\b0x[0-9a-fA-F]+\b', '', text)
    # Remove timestamps
    text = re.sub(r'\b\d{2}:\d{2}:\d{2}(\.\d+)?\b', '', text)
    # Remove dates
    text = re.sub(r'\b\d{4}-\d{2}-\d{2}\b', '', text)
    # Remove version numbers
    text = re.sub(r'\bv?\d+\.\d+(\.\d+)*\b', '', text)
    # Remove separator lines
    text = re.sub(r'[-=*_#~^|<>]{3,}', ' ', text)

    # Remove noisy lines
    lines = text.split('\n')
    clean_lines = []
    for line in lines:
        line = line.strip()
        if len(line) < 3:
            continue
        alpha_count = sum(c.isalpha() for c in line)
        if alpha_count < 3:
            continue
        if alpha_count / (len(line) + 1) < 0.3:
            continue
        clean_lines.append(line)

    text = ' '.join(clean_lines)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("⏳ Cleaning texts — please wait...")
df['text'] = df['text'].apply(clean_text)
print("✅ Text cleaning done!")
print(f"Avg words (after clean): {df['text'].str.split().str.len().mean():.0f}")

⏳ Cleaning texts — please wait...
✅ Text cleaning done!
Avg words (after clean): 46


In [ ]:
def truncate(text, max_words=100):
    return ' '.join(text.split()[:max_words])

df['text'] = df['text'].apply(truncate)

print("✅ Truncated to max 100 words!")
print(f"Avg words: {df['text'].str.split().str.len().mean():.0f}")
print(f"Max words: {df['text'].str.split().str.len().max()}")

✅ Truncated to max 100 words!
Avg words: 34
Max words: 100


In [ ]:
df = df[df['text'].str.split().str.len() >= 5].copy()
df = df.drop_duplicates(subset='text').copy()
df = df.reset_index(drop=True)

print(f"✅ After quality filter: {df.shape}")
print(df['priority'].value_counts())

✅ After quality filter: (33092, 10)
priority
Medium    12945
Low       10521
High       9626
Name: count, dtype: int64


In [ ]:
label_map = {'High': 0, 'Medium': 1, 'Low': 2}
df['priority_id'] = df['priority'].map(label_map)
df['source']      = 'gitbugs_core'

final = df[['text', 'priority', 'priority_id', 'source']].copy()

print("✅ Final columns set!")
print(final.head(3).to_string())

✅ Final columns set!
                                                                                                                                                                                                                                                                                                                                                                     text priority  priority_id        source
0  [macos] Save to PDF feature defaults filename to untitled.pdf instead of the title of the webpage. Steps to reproduce: 1. navigate to google.com or any webpage 1. press cmd-p after page loads 2. select save as pdf from the lower left dropdown. expected results: filename is the title of the page with .pdf appended. actual results: filename is populated with   Medium            1  gitbugs_core
1                                                                                                                                      Hit MOZ_CRASH(ElementAt(aIndex = 0, aLength = 0)

In [ ]:
wc = final['text'].str.split().str.len()

print("=" * 50)
print("   ✅ CORE_CLEAN — FINAL RESULTS")
print("=" * 50)
print(f"  Total rows     : {len(final):,}")
print(f"  Avg word count : {wc.mean():.0f}")
print(f"  Min word count : {wc.min()}")
print(f"  Max word count : {wc.max()}")
print()
print("  Priority distribution:")
print(final['priority'].value_counts().to_string())
print()
print("  Priority %:")
print((final['priority'].value_counts(normalize=True)*100).round(1).to_string())
print()
print("  Sample rows:")
for _, row in final.sample(3, random_state=42).iterrows():
    print(f"\n  [{row['priority']}] {row['text'][:120]}")
print()
print("=" * 50)

final.to_csv('core_clean.csv', index=False)
files.download('core_clean.csv')
print("✅ Downloaded: core_clean.csv")

   ✅ CORE_CLEAN — FINAL RESULTS
  Total rows     : 33,092
  Avg word count : 50
  Min word count : 5
  Max word count : 100

  Priority distribution:
priority
Medium    12945
Low       10521
High       9626

  Priority %:
priority
Medium    39.1
Low       31.8
High      29.1

  Sample rows:

  [Low] Intermittent /css/CSS2/normal-flow/max-height-094.xht | single tracking bug Reftest URL:**

  [Low] no automatic connection in jack audio kit Steps to reproduce: launch FF. Actual results: No automatic connection to jack

  [High] File descriptor limit is not loose enough Steps to reproduce: OS: Arch Linux Graphics: Intel Iris Xe (12th Gen) Desktop 



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Downloaded: core_clean.csv
